# Get the best genes of each group

Here we will go to all the sources of genes that we got. 
From the DDS we can have the log fold change the tstat and the pvalue
from the 

In [ ]:
# for each DDS, get the top 10 tstats values
# for each model weigth take also the top 10 abs_coef

In [1]:
from os import listdir
from os.path import isfile, join
import pandas as pd

In [2]:
dds_path = "data/DDS"
model_weigth_path = "data/model_weigths"

In [3]:
dds_files = [f for f in listdir(dds_path) if isfile(join(dds_path, f))]
model_weigth_files = [f for f in listdir(model_weigth_path) if isfile(join(model_weigth_path, f))]


In [4]:
model_weigth_files

['feature_importance_smote_catboost_7b_gene_ranking_07.csv',
 'RNAs_z_Combat_gene_ranking_40.csv',
 'feature_importance_smote_catboost_7b_gene_ranking_40.csv']

In [5]:
ensembl_to_symbol = pd.read_csv('data/ensmbl_list.csv', index_col = 0)
ensembl_to_symbol.head()
ensembl_to_symbol_dic = ensembl_to_symbol.to_dict('index')

In [7]:
top_genes = {}
for dds in dds_files:
    dds_df = pd.read_csv(dds_path+"/"+dds, index_col=0)
    dds_df = dds_df.dropna()
    topn = dds_df.nlargest(5, 'stat')
    title = dds.replace('RNAseq_abundances_adjusted_combat_inmose_', "").replace("_DDS.csv", "")
    top_genes[title]=topn

In [8]:
for model in model_weigth_files:
    model_df = pd.read_csv(model_weigth_path+"/"+model, index_col=0)
    model_df = model_df.dropna()
    topn = model_df.nlargest(5, 'abs_coef')
    ensembl_genes = topn["Ensembl"]
    ensembl_genes= [gene.split('.')[0] for gene in ensembl_genes ]
    symbols = [ensembl_to_symbol_dic[ensemble]['geneSymbol'] for ensemble in ensembl_genes]
    topn["Ensembl"]=symbols
    topn = topn.set_index("Ensembl")
    title = model.replace('feature_importance_smote_', "").replace(".csv", "").replace("RNAs_z_Combat_gene_ranking", "Ridge").replace("7b_gene_ranking_","").replace("RNAseq_abundances_adjusted_combat_inmose_", "")
    top_genes[title]=topn

In [ ]:
top_genes

In [10]:
for key in top_genes:
    top_genes[key]['Source'] = key

In [ ]:
dds_df.columns.to_list()

In [ ]:
model_df.columns.to_list()

In [11]:
all_columns = model_df.columns.to_list()  +dds_df.columns.to_list()+["Source"]

In [12]:
all_columns

['Ensembl',
 'coef',
 'abs_coef',
 'baseMean',
 'log2FoldChange',
 'lfcSE',
 'stat',
 'pvalue',
 'padj',
 'Source']

In [13]:
aligned_dfs = []
for key, df in top_genes.items():
    df = df.reindex(columns=all_columns, fill_value=None)  # Add missing columns
    aligned_dfs.append(df)

# Concatenate all dataframes
big_df = pd.concat(aligned_dfs, ignore_index=False)
big_df

,Ensembl,coef,abs_coef,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,Source
SLN,NaN,NaN,NaN,1.275066e+06,21.046520,1.465154,14.364712,8.616396e-47,4.516404e-44,young.vs.middle_female
Unnamed: 31307,NaN,NaN,NaN,2.464843e+08,33.332836,2.555548,13.043322,6.936723e-39,3.520552e-36,young.vs.middle_female
GIMAP1-GIMAP5,NaN,NaN,NaN,4.477939e+07,30.464950,2.425394,12.560823,3.467066e-36,1.732125e-33,young.vs.middle_female
INO80B-WBP1,NaN,NaN,NaN,1.045173e+07,28.273516,2.297363,12.306945,8.310970e-35,4.088230e-32,young.vs.middle_female
AS3MT,NaN,NaN,NaN,4.105222e+07,30.988594,2.547544,12.164105,4.827083e-34,2.338502e-31,young.vs.middle_female
...,...,...,...,...,...,...,...,...,...,...
STUM,NaN,19.534839,19.534839,NaN,NaN,NaN,NaN,NaN,NaN,catboost_40
MTFR1,NaN,9.205338,9.205338,NaN,NaN,NaN,NaN,NaN,NaN,catboost_40
DAPK2,NaN,6.349829,6.349829,NaN,NaN,NaN,NaN,NaN,NaN,catboost_40
FBXO31,NaN,4.661575,4.661575,NaN,NaN,NaN,NaN,NaN,NaN,catboost_40


In [14]:
big_df.to_csv("Top_5_genes_all_categories.csv")

In [15]:
big_df.index.value_counts()


AK1               6
ADA               4
OR51M1            4
BCORP1            4
GLB1L             4
                 ..
RECQL             1
Unnamed: 32706    1
SUSD6             1
NBEAP2            1
CLIC6             1
Name: count, Length: 65, dtype: int64

In [16]:
index_counts = big_df.index.value_counts()

# Filter for indices with a count greater than 1
repeated_indices = index_counts[index_counts > 1].index

# Get rows where the index is repeated
rows_with_repeated_indices = big_df.loc[big_df.index.isin(repeated_indices)].sort_index()

rows_with_repeated_indices["Source"]

AANAT                                 MO
AANAT                      middle.vs.old
ADA                 male.vs.female_young
ADA                male.vs.female_Middle
ADA                   male.vs.female_Old
ADA                 male.vs.female_Young
AK1                   male.vs.female_Old
AK1                male.vs.female_Middle
AK1                 male.vs.female_Young
AK1                   middle.vs.old_male
AK1                male.vs.female_middle
AK1                 male.vs.female_young
BCORP1              male.vs.female_Young
BCORP1              male.vs.female_young
BCORP1             male.vs.female_Middle
BCORP1                male.vs.female_Old
CLIC4               middle.vs.old_female
CLIC4                young.vs.old_female
CLIC4                 male.vs.female_old
GIMAP1-GIMAP5            young.vs.middle
GIMAP1-GIMAP5     young.vs.middle_female
GLB1L               male.vs.female_Young
GLB1L               male.vs.female_young
GLB1L                 male.vs.female_Old
GLB1L           